# Diabetes status classification — Small neural network

**Dataset:** `diabetes_binary_5050split_health_indicators_BRFSS2015.csv` from DTU Learn. Set `DATA_PATH` below to its local location. Run cells in order. Install dependencies if needed: `pip install numpy pandas scikit-learn matplotlib seaborn` plus `torch captum`.

The label `Diabetes_binary` denotes reported (pre)diabetes **at survey time**. The dataset was deliberately balanced 50/50. A predicted probability is therefore a model score for this sampled population, **not** an individual's future absolute risk. Feature contributions describe the model, not causal effects or medical advice. Keep the held-out test set untouched while choosing settings.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             roc_auc_score, average_precision_score, RocCurveDisplay,
                             brier_score_loss)
from sklearn.inspection import permutation_importance

DATA_PATH = Path('diabetes_binary_5050split_health_indicators_BRFSS2015.csv')
SEED = 42
if not DATA_PATH.is_file():
    raise FileNotFoundError(f'Set DATA_PATH to the CSV downloaded from DTU Learn: {DATA_PATH.resolve()}')
df = pd.read_csv(DATA_PATH)
target = 'Diabetes_binary'
if target not in df:
    raise KeyError(f'Missing {target}; columns: {df.columns.tolist()}')
if df.columns.duplicated().any():
    raise ValueError('Duplicate column names')
X = df.drop(columns=target).apply(pd.to_numeric, errors='raise')
y = df[target].astype(int)
assert set(y.unique()) == {0, 1}, 'Expected binary label 0/1'
print('Rows, features:', X.shape, '| positive fraction:', round(y.mean(), 3))
print('Missing values:', int(X.isna().sum().sum()))
display(X.head())

# Identical split in both notebooks. Split before fitting any imputation/scaling.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED)
features = X.columns.tolist()


## Prepare inputs and train

A training-only median imputer and standard scaler prevent leakage. A small network with one hidden layer outputs a logit. BCEWithLogitsLoss handles the sigmoid stably.

In [ ]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
preprocess = make_pipeline(SimpleImputer(strategy='median'), StandardScaler())
Z_train = preprocess.fit_transform(X_train).astype('float32')
Z_test = preprocess.transform(X_test).astype('float32')
# Validation comes exclusively from the training portion, with the same seed.
Z_fit, Z_val, y_fit, y_val = train_test_split(
    Z_train, y_train.to_numpy(), test_size=0.20,
    stratify=y_train, random_state=SEED)
loader = DataLoader(TensorDataset(torch.from_numpy(Z_fit),
                                  torch.from_numpy(y_fit.astype('float32'))),
                    batch_size=256, shuffle=True)
net = nn.Sequential(nn.Linear(len(features), 32), nn.ReLU(), nn.Linear(32, 1)).to(device)
optimizer = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.BCEWithLogitsLoss()
history = []
for epoch in range(30):
    net.train()
    for xb, yb in loader:
        optimizer.zero_grad()
        loss = loss_fn(net(xb.to(device)).squeeze(1), yb.to(device))
        loss.backward()
        optimizer.step()
    net.eval()
    with torch.no_grad():
        val_logits = net(torch.from_numpy(Z_val).to(device)).squeeze(1)
        val_loss = loss_fn(val_logits, torch.from_numpy(y_val.astype('float32')).to(device)).item()
    history.append(val_loss)
print('Device:', device, '| final validation loss:', round(history[-1], 4))
plt.plot(range(1, len(history)+1), history)
plt.xlabel('Epoch'); plt.ylabel('Validation BCE loss'); plt.show()


## Held-out evaluation

The fixed 30 epochs are an illustrative baseline. Choose any tuning or early stopping using validation data only.

In [ ]:
net.eval()
with torch.no_grad():
    probabilities = torch.sigmoid(net(torch.from_numpy(Z_test).to(device))).cpu().numpy().ravel()
def evaluate(y_true, probability, threshold=0.5):
    prediction = (probability >= threshold).astype(int)
    print(classification_report(y_true, prediction, target_names=['no (pre)diabetes', '(pre)diabetes']))
    print(f'ROC AUC: {roc_auc_score(y_true, probability):.3f}')
    print(f'Average precision: {average_precision_score(y_true, probability):.3f}')
    print(f'Brier score (balanced sample only): {brier_score_loss(y_true, probability):.3f}')
    ConfusionMatrixDisplay.from_predictions(y_true, prediction, display_labels=['No', 'Yes'], cmap='Blues')
    plt.title('Held-out test set, threshold 0.5'); plt.show()
    RocCurveDisplay.from_predictions(y_true, probability); plt.show()

evaluate(y_test, probabilities)


## Global explanation: permutation importance

The wrapper applies the training-only preprocessing before calling the network.

In [ ]:
class NetworkClassifier:
    classes_ = np.array([0, 1])
    def predict_proba(self, X_input):
        z = preprocess.transform(X_input).astype('float32')
        net.eval()
        with torch.no_grad():
            p = torch.sigmoid(net(torch.from_numpy(z).to(device))).cpu().numpy().ravel()
        return np.column_stack([1-p, p])

model = NetworkClassifier()
# Global audit: drop in held-out ROC AUC after shuffling one feature.
# Correlated inputs may share importance; shuffling can produce unrealistic combinations.
sample = X_test.sample(n=min(2000, len(X_test)), random_state=SEED)
result = permutation_importance(model, sample, y_test.loc[sample.index],
                                scoring='roc_auc', n_repeats=5, random_state=SEED, n_jobs=1)
importance = pd.Series(result.importances_mean, index=features).sort_values()
importance.tail(12).plot.barh(figsize=(8, 5), xlabel='Mean decrease in ROC AUC')
plt.title('Permutation importance on held-out sample'); plt.tight_layout(); plt.show()
display(importance.sort_values(ascending=False).head(15).to_frame('mean_auc_drop'))


## Local explanation: Integrated Gradients

Attributions are for the **logit**, relative to a median-feature baseline transformed with the training scaler. Positive values raise the logit relative to that baseline. Correlation, the chosen baseline and model error limit interpretation. These values are not causal intervention effects.

In [ ]:
from captum.attr import IntegratedGradients

row_position = 0
row = X_test.iloc[[row_position]]
z = torch.from_numpy(preprocess.transform(row).astype('float32')).to(device)
# Training medians correspond approximately to zero after imputation and scaling.
baseline_raw = pd.DataFrame([X_train.median()], columns=features)
baseline = torch.from_numpy(preprocess.transform(baseline_raw).astype('float32')).to(device)
net.eval()
attr, delta = IntegratedGradients(net).attribute(
    z, baselines=baseline, n_steps=100, return_convergence_delta=True)
local = pd.Series(attr.detach().cpu().numpy().ravel(), index=features).sort_values()
pd.concat([local.head(6), local.tail(6)]).plot.barh(figsize=(8, 5))
plt.axvline(0, color='black', lw=0.8)
plt.xlabel('Contribution to logit vs. median baseline'); plt.tight_layout(); plt.show()
print('Model score:', round(model.predict_proba(row)[0, 1], 3))
print('True survey label:', int(y_test.iloc[row_position]))
print('IG convergence delta:', float(delta.detach().cpu().item()))


## Error audit

In [ ]:
audit = X_test.copy()
audit['actual'] = y_test
audit['score'] = probabilities
audit['predicted'] = (probabilities >= 0.5).astype(int)
display(audit.loc[(audit.actual == 1) & (audit.predicted == 0)].sort_values('score').head(10))
display(audit.loc[(audit.actual == 0) & (audit.predicted == 1)].sort_values('score', ascending=False).head(10))
